# Load Packages

In [ ]:
import numpy as np
import pandas as pd
import random
from scipy.sparse import csr_matrix, lil_matrix
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import roc_auc_score

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# Load Datasets

In [ ]:

news_path = '/content/drive/MyDrive/features/news_features/news_features.csv'
news = pd.read_csv(news_path)

#train, val, test set
train = pd.read_csv('/content/drive/MyDrive/features/user_features/train_user_features.csv')
val = pd.read_csv('/content/drive/MyDrive/features/user_features/val_user_features.csv')
test = pd.read_csv('/content/drive/MyDrive/features/user_features/test_user_features.csv')

train_context = pd.read_csv('/content/drive/MyDrive/features/context_features/train_context.csv')


In [ ]:
#news dataset
print(news.head(10))


   news_id                subclustered_category  ctr_norm  bert_emb_0  \
0    26629                      lifestyleroyals -0.038402   -0.006567   
1    45241                           weightloss -0.038402    0.013164   
2    49591  CounterTerrorism_MiddleEastConflict -0.038402   -0.018924   
3    28765                       health_general -0.038402    0.047500   
4    28005                              medical -0.038402    0.000916   
5    31230                      NFL_StarPlayers -0.038402    0.015787   
6    43056                 ClimateChangeImpacts -0.038402    0.010033   
7    47796         IntlHealthRisk_UnusualEvents -0.038402    0.035753   
8    36210                               gaming -0.038402   -0.020225   
9     6638             newsscienceandtechnology -0.038402    0.032412   

   bert_emb_1  bert_emb_2  bert_emb_3  bert_emb_4  bert_emb_5  bert_emb_6  \
0    0.030011    0.041711    0.008560    0.023631    0.013891    0.019353   
1    0.027189    0.018828    0.059265   -0

In [ ]:
#train_context set
print(train_context.head(10))

   impression_id  user_id                 time  \
0          35263        1  2019-11-09 05:59:43   
1              1        1  2019-11-11 09:05:58   
2          20676        2  2019-11-09 05:28:51   
3           6673        2  2019-11-09 18:21:10   
4          97245        2  2019-11-10 05:43:35   
5         151853        2  2019-11-10 07:13:56   
6         135354        2  2019-11-10 07:16:53   
7          10938        2  2019-11-10 17:37:46   
8         124765        2  2019-11-10 18:02:10   
9          57351        2  2019-11-11 16:50:30   

                                             history  pos_id  \
0  [26659, 7649, 10157, 13853, 11735, 5063, 22437...     515   
1  [26659, 7649, 10157, 13853, 11735, 5063, 22437...   17918   
2  [19437, 27124, 48998, 22350, 37334, 22912, 447...   44926   
3  [19437, 27124, 48998, 22350, 37334, 22912, 447...   49940   
4  [19437, 27124, 48998, 22350, 37334, 22912, 447...    4154   
5  [19437, 27124, 48998, 22350, 37334, 22912, 447...   38476   
6

In [ ]:
#val set
print(val.head(10))

   impression_id  user_id                 time  \
0              9        9  2019-11-13 10:13:02   
1             12       12  2019-11-13 09:22:33   
2             19       19  2019-11-13 11:18:10   
3             20       20  2019-11-13 10:24:24   
4             23       23  2019-11-13 17:53:14   
5             27       27  2019-11-13 08:43:01   
6             28       28  2019-11-13 13:14:00   
7             39       39  2019-11-13 10:25:26   
8             44       44  2019-11-13 13:14:43   
9             46       46  2019-11-13 08:31:41   

                                             history         pos_ids  \
0  [21730, 35908, 35939, 2162, 7695, 42513, 21573...         [35647]   
1  [20858, 41662, 23282, 26659, 10405, 762, 7205,...         [31263]   
2                [42577, 11383, 48445, 15612, 27167]  [13656, 25555]   
3  [7257, 41847, 27409, 23815, 34008, 8026, 40607...         [34882]   
4  [38433, 39971, 35908, 39923, 21622, 35345, 431...  [40578, 37299]   
5  [32848, 33615,

## User-Based CF Model



In [ ]:
#create the user-item matrix using memory
def build_user_item_matrix(train_user: pd.DataFrame, train_context: pd.DataFrame):
    # reading memory from train_user
    memory_exploded = (
        train_user[["user_id", "history"]]
        .drop_duplicates(subset="user_id")
        .explode("history")
        .dropna(subset=["history"])
        .rename(columns={"history": "news_id"})
    )
    memory_exploded["news_id"] = memory_exploded["news_id"].astype(int)

    # impression clicks from train_context — pos_id is the clicked article
    impression_clicks = train_context[["user_id", "pos_id"]].copy()
    impression_clicks = impression_clicks.rename(columns={"pos_id": "news_id"})
    impression_clicks["news_id"] = impression_clicks["news_id"].astype(int)

    all_user_ids    = train_user["user_id"].unique()
    all_article_ids = pd.concat([
        impression_clicks["news_id"],
        memory_exploded["news_id"]
    ]).unique()

    user_index    = {u: i for i, u in enumerate(all_user_ids)}
    article_index = {a: i for i, a in enumerate(all_article_ids)}

    n_users    = len(user_index)
    n_articles = len(article_index)

    matrix = lil_matrix((n_users, n_articles), dtype=np.float32)

    # source 1 — impression clicks
    for _, row in impression_clicks.iterrows():
        if row["user_id"] in user_index and row["news_id"] in article_index:
            u = user_index[row["user_id"]]
            a = article_index[row["news_id"]]
            matrix[u, a] = 1.0

    # source 2 — reading memory clicks
    for _, row in memory_exploded.iterrows():
        if row["user_id"] in user_index and row["news_id"] in article_index:
            u = user_index[row["user_id"]]
            a = article_index[row["news_id"]]
            matrix[u, a] = 1.0

    matrix = matrix.tocsr()

    print(f"User-item matrix: {n_users:,} users x {n_articles:,} articles")
    print(f"Non-zero entries: {matrix.nnz:,}")
    print(f"Density: {matrix.nnz / (n_users * n_articles):.6f}")
    return matrix, user_index, article_index

In [ ]:
#compute cosine similarity of users and top-k neighbours
def compute_similarity_and_top_k(
    matrix     : csr_matrix,
    k          : int = 20,
    batch_size : int = 500,
):
    n_users       = matrix.shape[0]
    top_k_indices = np.zeros((n_users, k), dtype=np.int32)
    top_k_sims    = np.zeros((n_users, k), dtype=np.float32)

    print(f"Computing similarity + top-{k} neighbours for {n_users:,} users...")

    for start in range(0, n_users, batch_size):
        end = min(start + batch_size, n_users)

        # [batch_size, n_users] — only lives in memory for this iteration
        batch_sim = cosine_similarity(matrix[start:end], matrix)

        # zero self-similarity within this batch
        for i in range(end - start):
            batch_sim[i, start + i] = 0.0

        # top-K per row
        # and avoids sorting the full row
        for i, sim_row in enumerate(batch_sim):
            top_k = np.argpartition(sim_row, -k)[-k:]
            top_k = top_k[np.argsort(-sim_row[top_k])]
            top_k_indices[start + i] = top_k
            top_k_sims[start + i]    = sim_row[top_k]

        if start % 5000 == 0:
            print(f"  {start:,} / {n_users:,} done")

    print("Done.")
    return top_k_indices, top_k_sims

In [ ]:
#calculate score for each neighbour weighted by similarity
def score_candidates(
    target_user_idx : int,
    candidate_idxs  : list,
    top_k_indices   : np.ndarray,
    top_k_sims      : np.ndarray,
    matrix          : csr_matrix,
) -> np.ndarray:
    neighbour_idxs = top_k_indices[target_user_idx]   # [K]
    neighbour_sims = top_k_sims[target_user_idx]      # [K]

    # [K, n_candidates]
    neighbour_interactions = matrix[neighbour_idxs][:, candidate_idxs].toarray()

    weighted_sum = (neighbour_sims[:, np.newaxis] * neighbour_interactions).sum(axis=0)
    total_weight = np.abs(neighbour_sims).sum()

    if total_weight == 0:
        return np.zeros(len(candidate_idxs), dtype=np.float32)

    return (weighted_sum / total_weight).astype(np.float32)

In [ ]:
#performance metrics (MRR, NDCG, MAP)
def compute_mrr(labels: list, scores: list) -> float:
    """Mean Reciprocal Rank."""
    ranked = [l for _, l in sorted(zip(scores, labels), reverse=True)]
    for rank, label in enumerate(ranked, start=1):
        if label == 1:
            return 1.0 / rank
    return 0.0


def compute_ndcg(labels: list, scores: list, k: int) -> float:
    ranked = [l for _, l in sorted(zip(scores, labels), reverse=True)][:k]
    dcg    = sum((2**l - 1) / np.log2(r + 2) for r, l in enumerate(ranked))
    ideal  = sorted(labels, reverse=True)[:k]
    idcg   = sum((2**l - 1) / np.log2(r + 2) for r, l in enumerate(ideal))
    return dcg / idcg if idcg > 0 else 0.0

def compute_map(labels: list, scores: list) -> float:
    ranked_labels = [l for _, l in sorted(zip(scores, labels), reverse=True)]

    num_positives = sum(labels)
    if num_positives == 0:
        return 0.0

    precisions = []
    hits = 0
    for rank, label in enumerate(ranked_labels, start=1):
        if label == 1:
            hits += 1
            precisions.append(hits / rank)

    return sum(precisions) / num_positives

In [ ]:
#evaluation of model performance
def evaluate(
    eval_df       : pd.DataFrame,
    matrix        : csr_matrix,
    top_k_sims    : np.ndarray,
    top_k_indices : np.ndarray,
    user_index    : dict,
    article_index : dict,
) -> dict:
    aucs, mrrs, ndcg5s, ndcg10s, maps = [], [], [], [], []
    skipped = 0

    for _, row in eval_df.iterrows():

        user_id = row["user_id"]

        # skip cold-start users not seen in training
        if user_id not in user_index:
            skipped += 1
            continue

        pos_ids        = list(row["pos_ids"])
        neg_ids        = list(row["neg_ids"])
        all_candidates = pos_ids + neg_ids
        labels         = [1] * len(pos_ids) + [0] * len(neg_ids)

        # shuffle to remove position bias
        combined       = list(zip(all_candidates, labels))
        random.shuffle(combined)
        all_candidates, labels = zip(*combined)
        all_candidates = list(all_candidates)
        labels         = list(labels)

        # vocab filter — keep only articles seen in training
        valid_pairs = [
            (n, l) for n, l in zip(all_candidates, labels)
            if n in article_index
        ]

        # need at least one positive and one negative after filtering
        pos_kept = sum(1 for _, l in valid_pairs if l == 1)
        neg_kept = sum(1 for _, l in valid_pairs if l == 0)

        if pos_kept == 0 or neg_kept == 0:
            skipped += 1
            continue

        valid_news_ids, labels = zip(*valid_pairs)
        labels         = list(labels)
        candidate_idxs = [article_index[n] for n in valid_news_ids]

        scores = score_candidates(
            user_index[user_id],
            candidate_idxs,
            top_k_indices,
            top_k_sims,
            matrix,
        ).tolist()

        if len(set(labels)) > 1:
            aucs.append(roc_auc_score(labels, scores))
        mrrs.append(compute_mrr(labels, scores))
        ndcg5s.append(compute_ndcg(labels, scores, k=5))
        ndcg10s.append(compute_ndcg(labels, scores, k=10))
        maps.append(compute_map(labels, scores))

    print(f"Skipped {skipped:,} impressions (cold-start users, cold-start items or no positives/negatives)")
    return {
        "AUC"    : float(np.mean(aucs))    if aucs    else 0.0,
        "MRR"    : float(np.mean(mrrs))    if mrrs    else 0.0,
        "MAP"    : float(np.mean(maps))    if maps    else 0.0,
        "nDCG@5" : float(np.mean(ndcg5s)) if ndcg5s  else 0.0,
        "nDCG@10": float(np.mean(ndcg10s))if ndcg10s else 0.0,
    }

In [ ]:
#function to recommend top 10 news articles (excludes articles already seen)
def recommend(
    user_id       : int,
    user_index    : dict,
    article_index : dict,
    top_k_indices : np.ndarray,
    top_k_sims    : np.ndarray,
    matrix        : csr_matrix,
    top_n         : int = 10,
) -> list:
    if user_id not in user_index:
        print(f"User {user_id} not seen in training — cold start.")
        return []

    target_user_idx   = user_index[user_id]
    seen_article_idxs = set(matrix[target_user_idx].nonzero()[1])
    all_article_idxs  = list(article_index.values())

    scores = score_candidates(
        target_user_idx, all_article_idxs,
        top_k_indices, top_k_sims, matrix
    )
    idx_to_news = {v: k for k, v in article_index.items()}

    ranked = sorted(
        [
            (idx_to_news[idx], float(score))
            for idx, score in zip(all_article_idxs, scores)
            if idx not in seen_article_idxs
        ],
        key=lambda x: x[1],
        reverse=True,
    )
    return ranked[:top_n]




## RUN user-based CF

In [ ]:
import ast

# convert history from string to list
train["history"] = train["history"].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) else x
)

# also convert for train_context if needed
train_context["history"] = train_context["history"].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) else x
)

for df in [val, test]:
    df["history"] = df["history"].apply(
        lambda x: ast.literal_eval(x) if isinstance(x, str) else x
    )
    df["pos_ids"] = df["pos_ids"].apply(
        lambda x: ast.literal_eval(x) if isinstance(x, str) else x
    )
    df["neg_ids"] = df["neg_ids"].apply(
        lambda x: ast.literal_eval(x) if isinstance(x, str) else x
    )

In [ ]:
matrix, user_index, article_index = build_user_item_matrix(train, train_context)

User-item matrix: 40,148 users x 35,982 articles
Non-zero entries: 936,005
Density: 0.000648


In [ ]:
top_k_indices, top_k_sims = compute_similarity_and_top_k(matrix, k=200)


Computing similarity + top-200 neighbours for 40,148 users...
  0 / 40,148 done
  5,000 / 40,148 done
  10,000 / 40,148 done
  15,000 / 40,148 done
  20,000 / 40,148 done
  25,000 / 40,148 done
  30,000 / 40,148 done
  35,000 / 40,148 done
  40,000 / 40,148 done
Done.


In [ ]:
#evaluate on validation set
val_results = evaluate(val, matrix, top_k_sims, top_k_indices, user_index, article_index)
for metric, value in val_results.items():
    print(f"  {metric}: {value:.4f}")

Skipped 21,411 impressions (cold-start users, cold-start items or no positives/negatives)
  AUC: 0.5525
  MRR: 0.4841
  MAP: 0.4811
  nDCG@5: 0.5097
  nDCG@10: 0.5743


In [ ]:
#evaluate on test set
test_results = evaluate(test, matrix, top_k_sims, top_k_indices, user_index, article_index)
for metric, value in test_results.items():
    print(f"  {metric}: {value:.4f}")

Skipped 27,372 impressions (cold-start users, cold-start items or no positives/negatives)
  AUC: 0.5680
  MRR: 0.6338
  MAP: 0.6338
  nDCG@5: 0.6836
  nDCG@10: 0.7159


In [ ]:
#recommend
user_recommend = int(train["user_id"].iloc[0])
print(f"\nTop-10 recommendations for user {user_recommend}:")
recs = recommend(
    user_recommend, user_index, article_index,
    top_k_indices, top_k_sims, matrix
)
for news_id, score in recs:
    print(f"  news_id={news_id}  score={score:.4f}")


Top-10 recommendations for user 1:
  news_id=11383  score=0.1071
  news_id=34585  score=0.1036
  news_id=37947  score=0.0699
  news_id=22147  score=0.0525
  news_id=39158  score=0.0508
  news_id=43715  score=0.0480
  news_id=16081  score=0.0473
  news_id=6979  score=0.0464
  news_id=40667  score=0.0460
  news_id=23815  score=0.0456
